In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/Font_Dataset_Large.zip" -d "/content/dataset"

In [14]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from tqdm import tqdm
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Dense, Flatten

In [17]:
# --- Path dataset (Gambar 5.4) ---
arial_path = '/content/dataset/Font Dataset Large/Arial'
brandish_path = '/content/dataset/Font Dataset Large/Akzidenz Grotesk'
consolas_path = '/content/dataset/Font Dataset Large/Algerian'
didot_path = '/content/dataset/Font Dataset Large/Agency'

In [18]:
# --- Mengambil nama file (Gambar 5.5) ---
arial_filenames = os.listdir(arial_path)
brandish_filenames = os.listdir(brandish_path)
consolas_filenames = os.listdir(consolas_path)
didot_filenames = os.listdir(didot_path)

arial_filenames[:5]

['Image_2485.jpg',
 'Image_4887.jpg',
 'Image_3193.jpg',
 'Image_425.jpg',
 'Image_823.jpg']

In [22]:
# --- Fungsi load_image (Gambar 5.7) ---
def load_image(filenames, path):
    images = []
    for filename in tqdm(filenames):
        image = cv2.imread(path+'/'+filename)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        images.append(image)
    return images

In [23]:
# --- Load semua gambar (Gambar 5.8) ---
arial_images = load_image(arial_filenames, arial_path)
brandish_images = load_image(brandish_filenames, brandish_path)
consolas_images = load_image(consolas_filenames, consolas_path)
didot_images = load_image(didot_filenames, didot_path)

100%|██████████| 5000/5000 [00:00<00:00, 11769.18it/s]


In [24]:
# --- Mencari ukuran terbesar dan terkecil (Gambar 5.9) ---
heights = []
widths = []
for image in arial_images:
    height = image.shape[0]
    width = image.shape[1]
    heights.append(height)
    widths.append(width)
print('max_h: {}, min_h: {}'.format(max(heights), min(heights)))
print('max_w: {}, min_w: {}'.format(max(widths), min(widths)))

max_h: 32, min_h: 32
max_w: 248, min_w: 30


In [25]:
# --- Fungsi take_80_pixels (Gambar 5.10) ---
def take_80_pixels(images_list):
    new_images = []
    for image in images_list:
        if image.shape[1] >= 80:
            new_images.append(image[:, :80])
        else:
            pass
    return np.array(new_images)


In [26]:
# --- Menggunakan fungsi take_80_pixels (Gambar 5.12) ---
arial_80 = take_80_pixels(arial_images)
brandish_80 = take_80_pixels(brandish_images)
consolas_80 = take_80_pixels(consolas_images)
didot_80 = take_80_pixels(didot_images)

In [27]:
# --- Cek shape (Gambar 5.13) ---
print(arial_80.shape)
print(brandish_80.shape)
print(consolas_80.shape)
print(didot_80.shape)

(4079, 32, 80)
(4537, 32, 80)
(4771, 32, 80)
(1793, 32, 80)


In [28]:
# --- Menggabungkan array (Gambar 5.14) ---
all_images = np.append(arial_80, brandish_80, axis=0)
all_images = np.append(all_images, consolas_80, axis=0)
all_images = np.append(all_images, didot_80, axis=0)

In [29]:
# --- Reshape (Gambar 5.15) ---
all_images = all_images.reshape(all_images.shape[0], all_images.shape[1], all_images.shape[2], 1)
print(all_images.shape)
print(all_images.shape[0])
print(all_images.shape[1])
print(all_images.shape[2])


(15180, 32, 80, 1)
15180
32
80


In [30]:
# --- Membuat label (Gambar 5.16) ---
labels = [0]*len(arial_80) + \
         [1]*len(brandish_80) + \
         [2]*len(consolas_80) + \
         [3]*len(didot_80)
len(labels)



15180

In [31]:
# --- Train-test split (Gambar 5.17) ---
X_train, X_test, y_train, y_test = train_test_split(all_images, labels,
                                                     test_size=0.3,
                                                     random_state=88)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [32]:
# --- Menambahkan axis baru (Gambar 5.20) ---
y_train = y_train[:, np.newaxis]
y_test = y_test[:, np.newaxis]



In [35]:
# --- OneHotEncoder (Gambar 5.21) ---
one_hot_encoder = OneHotEncoder(sparse_output=False)
y_train_encoded = one_hot_encoder.fit_transform(y_train)
y_test_encoded = one_hot_encoder.transform(y_test)

In [36]:
# --- Cek label data train (Gambar 5.22) ---
print(y_train[:5])
print(y_train_encoded[:5])



[[1]
 [0]
 [3]
 [0]
 [2]]
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]]


In [37]:
# --- Menginisialisasi input_shape (Gambar 5.26) ---
input_shape = (all_images.shape[1], all_images.shape[2], 1)


In [38]:

# --- Membuat model CNN (Gambar 5.27) ---
model = Sequential()
model.add(Conv2D(32, kernel_size=(3,3), input_shape=input_shape, activation='relu'))
model.add(Conv2D(64, kernel_size=(3,3), activation='relu'))
model.add(MaxPool2D(2, 2))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dense(50, activation='relu'))
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
# --- Menampilkan summary model (Gambar 5.28) ---
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 78, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 76, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 38, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 34048)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     3,404,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,428,970 (13.08 MB)

 Trainable params: 3,428,970 (13.08 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:

# --- Compile model (Gambar 5.29) ---
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])

# --- Melatih model (Gambar 5.30) ---
history = model.fit(X_train, y_train_encoded,
                    epochs=3,
                    validation_data=(X_test, y_test_encoded))

Epoch 1/3
333/333 ━━━━━━━━━━━━━━━━━━━━ 144s 426ms/step - acc: 0.9663 - loss: 3.3144 - val_acc: 1.0000 - val_loss: 0.0012
Epoch 2/3
333/333 ━━━━━━━━━━━━━━━━━━━━ 208s 445ms/step - acc: 1.0000 - loss: 8.2156e-05 - val_acc: 1.0000 - val_loss: 4.8756e-06
Epoch 3/3
333/333 ━━━━━━━━━━━━━━━━━━━━ 138s 414ms/step - acc: 1.0000 - loss: 1.0284e-06 - val_acc: 1.0000 - val_loss: 1.9386e-06


In [41]:
# --- Memprediksi data test (Gambar 5.32) ---
predictions = model.predict(X_test)
predictions = np.argmax(predictions, axis=1)


143/143 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step


In [42]:
# --- Mengubah y_test menjadi array 1 dimensi (Gambar 5.33) ---
y_test = y_test.reshape(y_test.shape[0])

# --- Membuat confusion matrix (Gambar 5.33) ---
cm = confusion_matrix(y_test, predictions)
label_names = ['arial', 'brandish', 'consolas', 'didot']

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cbar=False,
            xticklabels=label_names, yticklabels=label_names)
plt.show()


In [43]:
# --- Mencari indeks misklasifikasi (Gambar 5.35) ---
misses = []
for i in range(len(predictions)):
    if y_test[i] != predictions[i]:
        misses.append(i)
print(misses)

[]


In [44]:
# --- Menampilkan foto dari kedua indeks misklasifikasi (Gambar 5.36) ---
for i, miss in enumerate(misses):
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7, 2), sharex=False)
    axes[0][0].imshow(X_test[miss].reshape(X_test.shape[1], X_test.shape[2]))
    axes[0][0].set_title('actual: ' + label_names[y_test[miss]])
    axes[0][0].set_visible(True)
    axes[1][0].set_visible(False)
    axes[0][1].set_title('predicted: ' + label_names[predictions[miss]])
    axes[0][1].set_visible(True)
    axes[1][1].set_visible(False)
    plt.show()
